# T2 — Semantic ICD linking (bge-m3 + reranker) trên Colab

Build embedding index cho ICD-10-CM rồi đo hit@k trên dev gold.

**Bước 1:** Runtime → Change runtime type → **T4 GPU** → Save.
**Bước 2:** Runtime → **Run all** (mỗi cell tự clone repo nếu thiếu — không cần token, repo public).

Xong gửi lại khối `=== ICD-10 (CHẨN_ĐOÁN) ===` cho P2.

In [ ]:
# 1) Kiểm tra GPU (phải thấy Tesla T4). Nếu lỗi -> chưa bật GPU ở Runtime.
!nvidia-smi -L

In [ ]:
# 2) Clone repo (PUBLIC, không cần token) nhánh Duckun + cài phụ thuộc.
import os
if not os.path.isdir('/content/viettel-medreason'):
    !git clone -b Duckun https://github.com/tienndat1904/viettel-medreason.git /content/viettel-medreason
%cd /content/viettel-medreason
!git pull
!git log --oneline -1
# rapidfuzz cần cho RxNorm matcher (Colab không có sẵn); FlagEmbedding cho bge-m3 + reranker
!pip -q install FlagEmbedding==1.3.2 rapidfuzz==3.10.1

In [ ]:
# 3) Build index (lần đầu tải bge-m3 ~2.3GB, ~vài phút trên T4). Hết VRAM -> thêm --batch-size 64
import os
if not os.path.isdir('/content/viettel-medreason'):
    !git clone -b Duckun https://github.com/tienndat1904/viettel-medreason.git /content/viettel-medreason
%cd /content/viettel-medreason
!python src/kb/build_icd_index.py

In [ ]:
# 4) Bật semantic + đo trên 30 file dev gold (embed query + rerank cần GPU).
import os
if not os.path.isdir('/content/viettel-medreason'):
    !git clone -b Duckun https://github.com/tienndat1904/viettel-medreason.git /content/viettel-medreason
%cd /content/viettel-medreason
!sed -i 's/backend: lexical/backend: semantic/' configs/config.yaml
!python src/eval/eval_linking.py

In [ ]:
# 5) (Tuỳ chọn) Lưu index vào Google Drive để phiên sau khỏi build lại.
%cd /content/viettel-medreason
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/viettel_kb
!cp data/kb/icd10cm_emb.npy data/kb/icd10cm_index_meta.parquet /content/drive/MyDrive/viettel_kb/
print('Đã lưu index vào Drive/MyDrive/viettel_kb')

## Gửi lại cho P2
Copy khối kết quả ở cell 4 (`=== ICD-10 ...` và `=== RxNorm ...`).